# Student Performance Prediction
## Notebook 1 — Data Cleaning & Preparation

**Author:** Abdifatah Muhlar
**Data Source:** UCI Machine Learning Repository — Student Performance Dataset
**Dataset:** student-mat.csv (Mathematics course, Portugal)

---

### Objective
This notebook handles the initial loading, inspection, and cleaning of the
UCI Student Performance dataset. The goal is to produce a clean, well-prepared
dataset ready for exploratory analysis and machine learning modeling.

### Dataset Description
The dataset contains 395 students from two Portuguese secondary schools,
with 30 features covering demographic information, family background,
study habits, social behavior, and academic performance. The target
variable is G3 — the final grade on a scale of 0 to 20.

### Prediction Target
We convert G3 into a binary classification target:
- Pass: G3 >= 10
- Fail: G3 < 10

This reflects the standard Portuguese grading system where 10 is the
minimum passing grade.

In [1]:
# ============================================================
# SECTION 1 — Import Libraries
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

print("Libraries loaded successfully")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

Libraries loaded successfully
Pandas version: 2.2.2
Numpy version: 2.0.2


---
## Section 2 — Load Dataset

In [2]:
# ============================================================
# SECTION 2 — Load Dataset
# ============================================================

# Load the dataset
df = pd.read_csv('student-mat.csv', sep=';')

print("Dataset loaded successfully")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nFirst 5 rows:")
df.head()

Dataset loaded successfully
Shape: 395 rows, 33 columns

First 5 rows:


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


---
## Section 3 — Data Inspection

Before cleaning, we thoroughly inspect the dataset to understand
its structure, data types, and check for any missing or anomalous values.

In [3]:
# ============================================================
# SECTION 3 — Data Inspection
# ============================================================

# Data types
print("DATA TYPES:")
print(df.dtypes)

# Missing values
print("\nMISSING VALUES:")
print(df.isnull().sum())

# Basic statistics for numeric columns
print("\nBASIC STATISTICS:")
print(df.describe())

# Categorical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"\nCategorical columns ({len(cat_cols)}):")
print(cat_cols)

# Numeric columns
num_cols = df.select_dtypes(include='number').columns.tolist()
print(f"\nNumeric columns ({len(num_cols)}):")
print(num_cols)

DATA TYPES:
school        object
sex           object
age            int64
address       object
famsize       object
Pstatus       object
Medu           int64
Fedu           int64
Mjob          object
Fjob          object
reason        object
guardian      object
traveltime     int64
studytime      int64
failures       int64
schoolsup     object
famsup        object
paid          object
activities    object
nursery       object
higher        object
internet      object
romantic      object
famrel         int64
freetime       int64
goout          int64
Dalc           int64
Walc           int64
health         int64
absences       int64
G1             int64
G2             int64
G3             int64
dtype: object

MISSING VALUES:
school        0
sex           0
age           0
address       0
famsize       0
Pstatus       0
Medu          0
Fedu          0
Mjob          0
Fjob          0
reason        0
guardian      0
traveltime    0
studytime     0
failures      0
schoolsup     0
famsup  

---
## Section 4 — Create Target Variable

We convert the continuous G3 grade into a binary pass/fail target:
- Pass (1): G3 >= 10
- Fail (0): G3 < 10

This reflects the standard Portuguese grading system where 10 is
the minimum passing grade, and allows us to frame this as a
binary classification problem.

In [5]:
# ============================================================
# SECTION 4 — Create Target Variable
# ============================================================

# Create binary pass/fail column
df['Pass'] = (df['G3'] >= 10).astype(int)

# Check distribution
pass_count = df['Pass'].value_counts()
pass_pct = df['Pass'].value_counts(normalize=True) * 100

print("TARGET VARIABLE DISTRIBUTION")
print("=" * 35)
print(f"Pass (1) : {pass_count[1]} students ({pass_pct[1]:.1f}%)")
print(f"Fail (0) : {pass_count[0]} students ({pass_pct[0]:.1f}%)")
print(f"Total    : {len(df)} students")
print(f"\nClass balance is {'acceptable' if pass_pct.min() > 30 else 'imbalanced — needs attention'}")

TARGET VARIABLE DISTRIBUTION
Pass (1) : 265 students (67.1%)
Fail (0) : 130 students (32.9%)
Total    : 395 students

Class balance is acceptable


---
## Section 5 — Encode Categorical Variables

Machine learning models require numeric input. We encode all 17
categorical columns using Label Encoding, converting text values
to numbers while preserving the information.

Examples:
- sex: F → 0, M → 1
- address: R → 0, U → 1
- higher: no → 0, yes → 1

In [6]:
# ============================================================
# SECTION 5 — Encode Categorical Variables
# ============================================================

le = LabelEncoder()

# Store encoding mappings for documentation
encoding_map = {}

cat_cols = df.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    df[col] = le.fit_transform(df[col])
    encoding_map[col] = dict(zip(le.classes_, le.transform(le.classes_)))

print("ENCODING COMPLETE")
print("=" * 40)
for col, mapping in encoding_map.items():
    print(f"{col}: {mapping}")

print(f"\nAll {len(cat_cols)} categorical columns encoded successfully")
print(f"\nDataset shape after encoding: {df.shape}")

ENCODING COMPLETE
school: {'GP': np.int64(0), 'MS': np.int64(1)}
sex: {'F': np.int64(0), 'M': np.int64(1)}
address: {'R': np.int64(0), 'U': np.int64(1)}
famsize: {'GT3': np.int64(0), 'LE3': np.int64(1)}
Pstatus: {'A': np.int64(0), 'T': np.int64(1)}
Mjob: {'at_home': np.int64(0), 'health': np.int64(1), 'other': np.int64(2), 'services': np.int64(3), 'teacher': np.int64(4)}
Fjob: {'at_home': np.int64(0), 'health': np.int64(1), 'other': np.int64(2), 'services': np.int64(3), 'teacher': np.int64(4)}
reason: {'course': np.int64(0), 'home': np.int64(1), 'other': np.int64(2), 'reputation': np.int64(3)}
guardian: {'father': np.int64(0), 'mother': np.int64(1), 'other': np.int64(2)}
schoolsup: {'no': np.int64(0), 'yes': np.int64(1)}
famsup: {'no': np.int64(0), 'yes': np.int64(1)}
paid: {'no': np.int64(0), 'yes': np.int64(1)}
activities: {'no': np.int64(0), 'yes': np.int64(1)}
nursery: {'no': np.int64(0), 'yes': np.int64(1)}
higher: {'no': np.int64(0), 'yes': np.int64(1)}
internet: {'no': np.int64(

---
## Section 6 — Feature Selection

We remove G1, G2, and G3 from the feature set for modeling.

Reason:
- G3 is our target variable — cannot be a feature
- G1 and G2 are interim grades from the same course
- Including G1 and G2 would make the model trivially accurate
  since they directly predict G3 — this is data leakage
- A truly useful model should predict performance from
  behavioral and demographic factors alone, before any
  grades are available

This makes the model genuinely useful for early intervention —
identifying at-risk students before exam results are known.

In [7]:
# ============================================================
# SECTION 6 — Feature Selection
# ============================================================

# Drop grade columns to prevent data leakage
# Keep all behavioral, demographic, and social features
features_to_drop = ['G1', 'G2', 'G3']

X = df.drop(columns=features_to_drop + ['Pass'])
y = df['Pass']

print("FEATURE SELECTION COMPLETE")
print("=" * 40)
print(f"Features used for modeling : {X.shape[1]}")
print(f"Target variable            : Pass (1) / Fail (0)")
print(f"Total samples              : {len(X)}")
print(f"\nFeature list:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2}. {col}")

FEATURE SELECTION COMPLETE
Features used for modeling : 30
Target variable            : Pass (1) / Fail (0)
Total samples              : 395

Feature list:
   1. school
   2. sex
   3. age
   4. address
   5. famsize
   6. Pstatus
   7. Medu
   8. Fedu
   9. Mjob
  10. Fjob
  11. reason
  12. guardian
  13. traveltime
  14. studytime
  15. failures
  16. schoolsup
  17. famsup
  18. paid
  19. activities
  20. nursery
  21. higher
  22. internet
  23. romantic
  24. famrel
  25. freetime
  26. goout
  27. Dalc
  28. Walc
  29. health
  30. absences


---
## Section 7 — Save Clean Dataset

In [8]:
# ============================================================
# SECTION 7 — Save Clean Dataset
# ============================================================

# Save full cleaned dataset
df.to_csv('student_clean.csv', index=False)

# Save features and target separately
X.to_csv('student_features.csv', index=False)
y.to_csv('student_target.csv', index=False)

import os
files = ['student_clean.csv', 'student_features.csv', 'student_target.csv']
print("SAVED FILES")
print("=" * 40)
for f in files:
    if os.path.exists(f):
        print(f"{f} — {os.path.getsize(f)} bytes — OK")

print(f"""
CLEANING SUMMARY
================
Original columns  : 33
Encoded columns   : 17 categorical → numeric
New column added  : Pass (binary target)
Final columns     : 34
Features for ML   : 30 (G1, G2, G3 excluded)
Missing values    : 0
Total samples     : 395
Output files      : student_clean.csv
                    student_features.csv
                    student_target.csv
""")

SAVED FILES
student_clean.csv — 28338 bytes — OK
student_features.csv — 24397 bytes — OK
student_target.csv — 795 bytes — OK

CLEANING SUMMARY
Original columns  : 33
Encoded columns   : 17 categorical → numeric
New column added  : Pass (binary target)
Final columns     : 34
Features for ML   : 30 (G1, G2, G3 excluded)
Missing values    : 0
Total samples     : 395
Output files      : student_clean.csv
                    student_features.csv
                    student_target.csv



In [12]:
from google.colab import files
files.download('student_clean.csv')
files.download('student_features.csv')
files.download('student_targiet.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
from google.colab import files
files.download('student_features.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>